# GPU multiple-regression RSA â€” steps 15, 15.3, 15.4, 15.5

Defaults to **EmoB humans**; set `SPECIE = 'D'` for dogs. Uses the existing `rsa_colab/pkg_EmoB` participant packages and `results_EmoB` step-1 ZIPs. The small regression support ZIP supplies target/control CSVs, masks, and current code. No workstation paths or full toolkit installation are needed in Colab.

Select **Runtime â†’ Change runtime type â†’ GPU**. GPU work uses float64 batched OLS and group reductions; NIfTI/ZIP reading and writing still run on the CPU. Reduce batch sizes if GPU memory is tight. Local scratch holds one participant, or one target's group inputs at a time. Check free local disk space for large human permutation sets.

In [ ]:
# Install only lightweight dependencies; use Colab's installed CUDA PyTorch.
!pip -q install nibabel pandas scipy
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose a GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change these paths for another dataset/package collection.
BASE = '/content/drive/MyDrive/rsa_colab'
DATASET = 'EmoB'
SPECIE = 'H'                 # 'H' humans; 'D' dogs
MODEL = 'basic-block'
DIS_METHOD = 'correlation'   # matches the visual-control matrices
REGRESSION_MODEL = 'visual_3'
PACKAGES_DIR = f'{BASE}/pkg_EmoB'
STEP1_RESULTS_DIR = f'{BASE}/results_EmoB'  # None: use bundled maps or recompute from betas
SUPPORT_ZIP = f'{BASE}/regression_support_{DATASET}_{REGRESSION_MODEL}.zip'
OUT_DIR = f'{BASE}/results_regression_{DATASET}'

MODELS = None               # every target in support ZIP, excluding controls
PARTICIPANTS = None         # every expected participant in support ZIP
STEPS = [15, 15.3, 15.4, 15.5]
REPS = 100
REPS_GROUP = 1000
SEED = 42                  # stable per participant/run/target/permutation
VOXEL_BATCH = 2048
PERMUTATION_BATCH = 8
GROUP_BATCH = 16
STEP1_BATCH = 256
FORCE = False              # overwrite compatible checkpoints for requested steps
WORK_ROOT = '/content/regression_work'

In [ ]:
# Bootstrap only the known code files from the support package.
from pathlib import Path
import json, zipfile, sys, importlib, shutil
support_path = Path(SUPPORT_ZIP)
if not support_path.is_file():
    raise FileNotFoundError(f'Upload the regression support ZIP first: {SUPPORT_ZIP}')
code_dir = Path('/content/regression_code')
code_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(support_path) as zf:
    support = json.loads(zf.read('regression_manifest.json'))
    for filename in ('gpu_rsa.py', 'gpu_regression.py', 'run_colab_regression.py'):
        (code_dir / filename).write_bytes(zf.read('code/' + filename))
sys.path.insert(0, str(code_dir))
for name in ('gpu_rsa', 'gpu_regression', 'run_colab_regression'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import gpu_regression, run_colab_regression
print('Regression runtime:', gpu_regression.VERSION)
if gpu_regression.VERSION != '1.1.0':
    raise RuntimeError('Expected runtime 1.1.0. Wait for the updated support ZIP to sync, then rerun this cell.')
print('Controls:', support['controls'])
print('Available targets:', support['models'])

In [ ]:
# Check analysis identity and full package coverage before starting.
for key, expected in [('dataset', DATASET), ('model', MODEL),
                      ('dis_method', DIS_METHOD), ('regression_model', REGRESSION_MODEL)]:
    if support[key] != expected:
        raise ValueError(f'Support ZIP {key}={support[key]!r}; expected {expected!r}')
expected_participants = PARTICIPANTS if PARTICIPANTS is not None else support['participants_by_species'].get(SPECIE)
if not expected_participants:
    raise ValueError(f'Support ZIP has no participants for {SPECIE}. Rebuild with --specie H D.')
packages = run_colab_regression.discover_packages(
    PACKAGES_DIR, specie=SPECIE, dataset=DATASET, model=MODEL,
    dis_method=DIS_METHOD, participants=expected_participants)
print(f'{len(packages)} participants; {sum(len(m["runs"]) for _, m in packages.values())} runs')
print('Targets:', MODELS if MODELS is not None else support['models'])
print('Local disk free:', round(shutil.disk_usage('/content').free / 1e9, 1), 'GB')
print('GPU free:', round(torch.cuda.mem_get_info()[0] / 1e9, 1), 'GB')
print('Outputs:', OUT_DIR)
# Full group coverage is enforced. Use an explicit PARTICIPANTS list only when
# deliberately analyzing a subset; that subset becomes the group population.

In [ ]:
written = run_colab_regression.run_regression(
    PACKAGES_DIR, OUT_DIR, SUPPORT_ZIP,
    specie=SPECIE, dataset=DATASET, model=MODEL, dis_method=DIS_METHOD,
    models=MODELS, participants=PARTICIPANTS, regression_model=REGRESSION_MODEL,
    steps=STEPS, step1_results_dir=STEP1_RESULTS_DIR,
    reps=REPS, reps_group=REPS_GROUP, seed=SEED, work_root=WORK_ROOT,
    device='cuda', voxel_batch=VOXEL_BATCH, permutation_batch=PERMUTATION_BATCH,
    group_batch=GROUP_BATCH, step1_batch=STEP1_BATCH, force=FORCE)
print('New checkpoints:')
for path in written:
    print(path)
print('Requested steps complete.')

## What is calculated and saved

- **15:** standardized target beta/t/p maps from `[intercept, target, real controls]`.
- **15.3:** beta mean/std across participant/session/run maps, equally weighted as in `searchlight.py`. Participants with more runs contribute more maps.
- **15.4:** the same regression with only the target RDM's category labels permuted; controls and neural data remain fixed. It does not use saved RSA z-maps.
- **15.5:** each of `REPS_GROUP` draws selects one of the `REPS` fits independently per run, then averages the selected beta maps. Its std is spread across run maps, not the null-distribution std.

Observed outputs use `RSA_regression`; permutations use `RSA_regression_rnd`, with the CPU pipeline's filenames. There is one `result_regression_<target>_<species>-sub-NN.zip` per participant/target and one `result_regression_group_<target>_<species>.zip` per target. Version 1.1.0 logs each processing stage and saves a checkpoint after every target/run under OUT_DIR/run_checkpoints. Completed target/runs survive runtime disconnects; only unfinished target/runs are recomputed. Top-level participant and group ZIPs remain the final outputs to merge onto the workstation. Changing models, input archives, seed or repetition count invalidates affected checkpoints. Group-only steps require compatible participant result ZIPs in `OUT_DIR`.

Back on the workstation, merge the output ZIPs with:
```powershell
& 'C:\ProgramData\anaconda3\python.exe' tools\unpack_results.py 'G:\My Drive\rsa_colab\results_regression_EmoB'
```
Use `--replace` if replacing an earlier analysis. For another dataset, create new packages with `tools/create_regression_package.py`, upload the `packages/` folder and support ZIP, then update the settings above. Performance depends on GPU float64 throughput and Drive I/O; no speedup is assumed without benchmarking.